## Подготовка данных с использованием фреймворка Apache Spark

Подключим необходимые библиотеки.

In [1]:
import os
from pyspark.sql import SparkSession, DataFrame
from pyspark import SparkConf
from pyspark.sql.functions import (
    regexp_replace,
    regexp_extract_all,
    regexp_extract,
    col,
    lit,
    when,
    to_date,
    from_unixtime,
    split,
    trim,
    year,
    udf, 
    array,
    expr
)
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.sql import functions as F
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.ml import Pipeline

Сформируем объект конфигурации для `Apache Spark`, указав необходимые параметры.

In [2]:
def create_spark_configuration() -> SparkConf:
    """
    Создает и конфигурирует экземпляр SparkConf для приложения Spark.

    Returns:
        SparkConf: Настроенный экземпляр SparkConf.
    """

    conf = SparkConf()
    conf.setAppName("Load_DP")
    conf.setMaster("local[*]")
    conf.set("spark.driver.memory", "10g")           
    conf.set("spark.executor.memory", "15g")         
    conf.set("spark.memory.fraction", "0.8")        
    conf.set("spark.memory.storageFraction", "0.3") 
    
    # Настройки для больших данных
    conf.set("spark.sql.adaptive.enabled", "true")
    conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
    conf.set("spark.sql.adaptive.skew.enabled", "true")
    conf.set("spark.sql.shuffle.partitions", "100")  # Уменьшил партиции

    return conf

Создаём сам объект конфигурации.

In [3]:
conf = create_spark_configuration()

Создаём и выводим на экран сессию `Apache Spark`.

In [4]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark

Для исследования будем использовать датасет, расположенный по адресу https://ods.ai/competitions/dl-fintech-bki/data.

Указываем путь.

In [5]:
train_data_name = 'train_data'
train_path = f'../data/{train_data_name}/*.pq'


Заполняем датафрейм данными из файла.

In [6]:
train_df = spark.read.parquet(train_path)

Выводим фрагмент датафрейма на экран.

In [7]:
train_df.limit(20).toPandas().style\
    .set_properties(**{'text-align': 'left', 'max-width': '0', 'white-space': 'nowrap', 'overflow': 'hidden', 'text-overflow': 'ellipsis'})\
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])\
    .format(precision=2)

,id,rn,pre_since_opened,pre_since_confirmed,pre_pterm,pre_fterm,pre_till_pclose,pre_till_fclose,pre_loans_credit_limit,pre_loans_next_pay_summ,pre_loans_outstanding,pre_loans_total_overdue,pre_loans_max_overdue_sum,pre_loans_credit_cost_rate,pre_loans5,pre_loans530,pre_loans3060,pre_loans6090,pre_loans90,is_zero_loans5,is_zero_loans530,is_zero_loans3060,is_zero_loans6090,is_zero_loans90,pre_util,pre_over2limit,pre_maxover2limit,is_zero_util,is_zero_over2limit,is_zero_maxover2limit,enc_paym_0,enc_paym_1,enc_paym_2,enc_paym_3,enc_paym_4,enc_paym_5,enc_paym_6,enc_paym_7,enc_paym_8,enc_paym_9,enc_paym_10,enc_paym_11,enc_paym_12,enc_paym_13,enc_paym_14,enc_paym_15,enc_paym_16,enc_paym_17,enc_paym_18,enc_paym_19,enc_paym_20,enc_paym_21,enc_paym_22,enc_paym_23,enc_paym_24,enc_loans_account_holder_type,enc_loans_credit_status,enc_loans_credit_type,enc_loans_account_cur,pclose_flag,fclose_flag
0,0,1,18,9,2,3,16,10,11,3,3,0,2,11,6,16,5,4,8,1,1,1,1,1,16,2,17,1,1,1,0,0,3,3,3,3,3,3,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,4,1,0,0
1,0,2,18,9,14,14,12,12,0,3,3,0,2,11,6,16,5,4,8,1,1,1,1,1,16,2,17,1,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,4,1,3,4,1,0,0
2,0,3,18,9,4,8,1,11,11,0,5,0,2,8,6,16,5,4,8,1,1,1,1,1,15,2,17,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,4,1,2,3,1,1,1
3,0,4,4,1,9,12,16,7,12,2,3,0,2,4,6,16,5,4,8,0,1,1,1,1,16,2,17,1,1,1,1,0,0,0,0,0,0,0,0,0,0,1,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,1,1,0,0
4,0,5,5,12,15,2,11,12,10,2,3,0,2,4,6,16,5,4,8,1,1,1,1,1,16,2,17,1,1,1,0,0,0,0,0,0,0,3,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,4,1,0,0
5,0,6,5,0,11,8,12,11,4,2,3,0,2,4,6,16,5,4,8,1,1,1,1,1,9,5,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,3,4,3,3,3,4,1,2,3,1,0,1
6,0,7,3,9,1,2,12,14,15,5,3,0,2,3,6,16,5,4,8,1,1,1,1,1,16,2,17,1,1,1,0,0,0,0,0,0,0,0,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,4,1,0,0
7,0,8,2,9,2,3,12,14,15,5,3,0,2,13,6,16,5,4,8,1,1,1,1,1,16,2,17,1,1,1,0,0,3,3,3,3,3,3,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,4,1,0,0
8,0,9,1,9,11,13,14,8,2,5,1,0,2,11,6,16,5,4,8,1,1,1,1,1,1,2,17,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,3,3,3,3,3,3,4,3,3,3,4,1,2,4,1,0,0
9,0,10,7,9,2,10,8,8,16,4,2,0,2,11,6,16,5,4,8,1,1,1,1,1,15,2,17,0,1,1,0,0,0,0,0,0,3,3,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,2,4,1,0,0


In [8]:
train_df.printSchema()

root
 |-- id: long (nullable = true)
 |-- rn: long (nullable = true)
 |-- pre_since_opened: long (nullable = true)
 |-- pre_since_confirmed: long (nullable = true)
 |-- pre_pterm: long (nullable = true)
 |-- pre_fterm: long (nullable = true)
 |-- pre_till_pclose: long (nullable = true)
 |-- pre_till_fclose: long (nullable = true)
 |-- pre_loans_credit_limit: long (nullable = true)
 |-- pre_loans_next_pay_summ: long (nullable = true)
 |-- pre_loans_outstanding: long (nullable = true)
 |-- pre_loans_total_overdue: long (nullable = true)
 |-- pre_loans_max_overdue_sum: long (nullable = true)
 |-- pre_loans_credit_cost_rate: long (nullable = true)
 |-- pre_loans5: long (nullable = true)
 |-- pre_loans530: long (nullable = true)
 |-- pre_loans3060: long (nullable = true)
 |-- pre_loans6090: long (nullable = true)
 |-- pre_loans90: long (nullable = true)
 |-- is_zero_loans5: long (nullable = true)
 |-- is_zero_loans530: long (nullable = true)
 |-- is_zero_loans3060: long (nullable = true)
 |

Видно, что все столбцы датасета содержат long тип данных, что не соответствует ожиданиям. Выполним преобразования типов данных столбцов.

In [9]:
def transform_dataframe(data: DataFrame) -> DataFrame:
    """
    Оптимизирует типы данных для кредитного датафрейма с учетом семантики полей.
    Минимизирует использование памяти за счет правильных типов.
    
    Args:
        data (DataFrame): Исходный DataFrame со всеми полями.
    
    Returns:
        DataFrame: Оптимизированный DataFrame с правильными типами данных.
    """
    # Словарь преобразований для каждой колонки
    type_transformations = {
        # ID и порядковые номера
        "id": ("int", "Идентификатор клиента"),
        "rn": ("short", "Номер наблюдения (макс ~32k)"),
        
        # Временные интервалы (в днях)
        "pre_since_opened": ("short", "Дней с открытия (макс ~32k дней ~88 лет)"),
        "pre_since_confirmed": ("short", "Дней с подтверждения"),
        "pre_pterm": ("short", "Плановый срок (дней)"),
        "pre_fterm": ("short", "Фактический срок (дней)"),
        "pre_till_pclose": ("short", "Дней до планового закрытия"),
        "pre_till_fclose": ("short", "Дней до фактического закрытия"),
        
        # Денежные суммы и лимиты (могут быть большими)
        "pre_loans_credit_limit": ("float", "Кредитный лимит"),
        "pre_loans_next_pay_summ": ("float", "Следующий платеж"),
        "pre_loans_outstanding": ("float", "Текущая задолженность"),
        "pre_loans_total_overdue": ("float", "Общая просрочка"),
        "pre_loans_max_overdue_sum": ("float", "Максимальная просрочка в истории"),
        
        # Проценты и ставки
        "pre_loans_credit_cost_rate": ("float", "Ставка по кредиту (%)"),
        
        # Количества кредитов (небольшие числа)
        "pre_loans5": ("byte", "Просрочки до 5 дней (макс 127)"),
        "pre_loans530": ("byte", "Просрочки 5-30 дней"),
        "pre_loans3060": ("byte", "Просрочки 30-60 дней"),
        "pre_loans6090": ("byte", "Просрочки 60-90 дней"),
        "pre_loans90": ("byte", "Просрочки >90 дней"),
        
        # Бинарные флаги (0/1)
        "is_zero_loans5": ("byte", "Флаг отсутствия просрочек до 5 дней"),
        "is_zero_loans530": ("byte", "Флаг отсутствия просрочек 5-30 дней"),
        "is_zero_loans3060": ("byte", "Флаг отсутствия просрочек 30-60 дней"),
        "is_zero_loans6090": ("byte", "Флаг отсутствия просрочек 60-90 дней"),
        "is_zero_loans90": ("byte", "Флаг отсутствия серьезных просрочек"),
        "is_zero_util": ("byte", "Флаг нулевой утилизации"),
        "is_zero_over2limit": ("byte", "Флаг отсутствия превышения лимита"),
        "is_zero_maxover2limit": ("byte", "Флаг отсутствия макс. превышения"),
        "pclose_flag": ("byte", "Флаг планового закрытия"),
        "fclose_flag": ("byte", "Флаг фактического закрытия"),
        
        # Проценты утилизации (0-100+)
        "pre_util": ("float", "Утилизация кредитного лимита (%)"),
        "pre_over2limit": ("float", "Отношение просрочки к лимиту"),
        "pre_maxover2limit": ("float", "Максимальное отношение просрочки к лимиту"),
        
        # Категориальные признаки (one-hot encoded, небольшие числа)
        "enc_paym_0": ("byte", "Платежный паттерн 0"),
        "enc_paym_1": ("byte", "Платежный паттерн 1"),
        "enc_paym_2": ("byte", "Платежный паттерн 2"),
        "enc_paym_3": ("byte", "Платежный паттерн 3"),
        "enc_paym_4": ("byte", "Платежный паттерн 4"),
        "enc_paym_5": ("byte", "Платежный паттерн 5"),
        "enc_paym_6": ("byte", "Платежный паттерн 6"),
        "enc_paym_7": ("byte", "Платежный паттерн 7"),
        "enc_paym_8": ("byte", "Платежный паттерн 8"),
        "enc_paym_9": ("byte", "Платежный паттерн 9"),
        "enc_paym_10": ("byte", "Платежный паттерн 10"),
        "enc_paym_11": ("byte", "Платежный паттерн 11"),
        "enc_paym_12": ("byte", "Платежный паттерн 12"),
        "enc_paym_13": ("byte", "Платежный паттерн 13"),
        "enc_paym_14": ("byte", "Платежный паттерн 14"),
        "enc_paym_15": ("byte", "Платежный паттерн 15"),
        "enc_paym_16": ("byte", "Платежный паттерн 16"),
        "enc_paym_17": ("byte", "Платежный паттерн 17"),
        "enc_paym_18": ("byte", "Платежный паттерн 18"),
        "enc_paym_19": ("byte", "Платежный паттерн 19"),
        "enc_paym_20": ("byte", "Платежный паттерн 20"),
        "enc_paym_21": ("byte", "Платежный паттерн 21"),
        "enc_paym_22": ("byte", "Платежный паттерн 22"),
        "enc_paym_23": ("byte", "Платежный паттерн 23"),
        "enc_paym_24": ("byte", "Платежный паттерн 24"),
        
        # Категории кредитов (малые целые)
        "enc_loans_account_holder_type": ("byte", "Тип держателя счета"),
        "enc_loans_credit_status": ("byte", "Статус кредита"),
        "enc_loans_credit_type": ("byte", "Тип кредита"),
        "enc_loans_account_cur": ("byte", "Валюта счета"),
    }
    
    # Применяем преобразования
    for column_name, (column_type, description) in type_transformations.items():
        if column_name in data.columns:
            data = data.withColumn(column_name, col(column_name).cast(column_type))
    
    return data

In [10]:
train_df = transform_dataframe(train_df)

In [11]:
train_df.limit(20).toPandas().style\
    .set_properties(**{'text-align': 'left', 'max-width': '0', 'white-space': 'nowrap', 'overflow': 'hidden', 'text-overflow': 'ellipsis'})\
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])\
    .format(precision=2)

,id,rn,pre_since_opened,pre_since_confirmed,pre_pterm,pre_fterm,pre_till_pclose,pre_till_fclose,pre_loans_credit_limit,pre_loans_next_pay_summ,pre_loans_outstanding,pre_loans_total_overdue,pre_loans_max_overdue_sum,pre_loans_credit_cost_rate,pre_loans5,pre_loans530,pre_loans3060,pre_loans6090,pre_loans90,is_zero_loans5,is_zero_loans530,is_zero_loans3060,is_zero_loans6090,is_zero_loans90,pre_util,pre_over2limit,pre_maxover2limit,is_zero_util,is_zero_over2limit,is_zero_maxover2limit,enc_paym_0,enc_paym_1,enc_paym_2,enc_paym_3,enc_paym_4,enc_paym_5,enc_paym_6,enc_paym_7,enc_paym_8,enc_paym_9,enc_paym_10,enc_paym_11,enc_paym_12,enc_paym_13,enc_paym_14,enc_paym_15,enc_paym_16,enc_paym_17,enc_paym_18,enc_paym_19,enc_paym_20,enc_paym_21,enc_paym_22,enc_paym_23,enc_paym_24,enc_loans_account_holder_type,enc_loans_credit_status,enc_loans_credit_type,enc_loans_account_cur,pclose_flag,fclose_flag
0,0,1,18,9,2,3,16,10,11.00,3.00,3.00,0.00,2.00,11.00,6,16,5,4,8,1,1,1,1,1,16.00,2.00,17.00,1,1,1,0,0,3,3,3,3,3,3,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,4,1,0,0
1,0,2,18,9,14,14,12,12,0.00,3.00,3.00,0.00,2.00,11.00,6,16,5,4,8,1,1,1,1,1,16.00,2.00,17.00,1,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,4,1,3,4,1,0,0
2,0,3,18,9,4,8,1,11,11.00,0.00,5.00,0.00,2.00,8.00,6,16,5,4,8,1,1,1,1,1,15.00,2.00,17.00,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,4,1,2,3,1,1,1
3,0,4,4,1,9,12,16,7,12.00,2.00,3.00,0.00,2.00,4.00,6,16,5,4,8,0,1,1,1,1,16.00,2.00,17.00,1,1,1,1,0,0,0,0,0,0,0,0,0,0,1,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,1,1,0,0
4,0,5,5,12,15,2,11,12,10.00,2.00,3.00,0.00,2.00,4.00,6,16,5,4,8,1,1,1,1,1,16.00,2.00,17.00,1,1,1,0,0,0,0,0,0,0,3,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,4,1,0,0
5,0,6,5,0,11,8,12,11,4.00,2.00,3.00,0.00,2.00,4.00,6,16,5,4,8,1,1,1,1,1,9.00,5.00,4.00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,3,4,3,3,3,4,1,2,3,1,0,1
6,0,7,3,9,1,2,12,14,15.00,5.00,3.00,0.00,2.00,3.00,6,16,5,4,8,1,1,1,1,1,16.00,2.00,17.00,1,1,1,0,0,0,0,0,0,0,0,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,4,1,0,0
7,0,8,2,9,2,3,12,14,15.00,5.00,3.00,0.00,2.00,13.00,6,16,5,4,8,1,1,1,1,1,16.00,2.00,17.00,1,1,1,0,0,3,3,3,3,3,3,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,4,1,0,0
8,0,9,1,9,11,13,14,8,2.00,5.00,1.00,0.00,2.00,11.00,6,16,5,4,8,1,1,1,1,1,1.00,2.00,17.00,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,3,3,3,3,3,3,4,3,3,3,4,1,2,4,1,0,0
9,0,10,7,9,2,10,8,8,16.00,4.00,2.00,0.00,2.00,11.00,6,16,5,4,8,1,1,1,1,1,15.00,2.00,17.00,0,1,1,0,0,0,0,0,0,3,3,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,2,4,1,0,0


In [12]:
train_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- rn: short (nullable = true)
 |-- pre_since_opened: short (nullable = true)
 |-- pre_since_confirmed: short (nullable = true)
 |-- pre_pterm: short (nullable = true)
 |-- pre_fterm: short (nullable = true)
 |-- pre_till_pclose: short (nullable = true)
 |-- pre_till_fclose: short (nullable = true)
 |-- pre_loans_credit_limit: float (nullable = true)
 |-- pre_loans_next_pay_summ: float (nullable = true)
 |-- pre_loans_outstanding: float (nullable = true)
 |-- pre_loans_total_overdue: float (nullable = true)
 |-- pre_loans_max_overdue_sum: float (nullable = true)
 |-- pre_loans_credit_cost_rate: float (nullable = true)
 |-- pre_loans5: byte (nullable = true)
 |-- pre_loans530: byte (nullable = true)
 |-- pre_loans3060: byte (nullable = true)
 |-- pre_loans6090: byte (nullable = true)
 |-- pre_loans90: byte (nullable = true)
 |-- is_zero_loans5: byte (nullable = true)
 |-- is_zero_loans530: byte (nullable = true)
 |-- is_zero_loans3060: byte (nul

In [13]:
class PySparkCountAggregator:
    """Unified aggregator for categorical + numeric features by id for Spark GBT"""

    def __init__(self, categorical_columns):
        self.categorical_columns = categorical_columns

    def _encode_categories(self, df):
        """Fit & apply one-hot encoders"""
        stages = []
        encoded_cols = []

        for c in self.categorical_columns:
            idx = StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
            ohe = OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
            stages += [idx, ohe]
            encoded_cols.append(f"{c}_ohe")

        self.model = Pipeline(stages=stages).fit(df)
        df = self.model.transform(df)
        self.cat_ohe_cols = encoded_cols
        return df

    def fit_transform(self, df):
        df = self._encode_categories(df)
        return self._aggregate(df)

    def transform(self, df):
        assert hasattr(self, "model"), "Call fit_transform() first"
        df = self.model.transform(df)
        return self._aggregate(df)

    def _aggregate(self, df):
        """Aggregate everything by id"""
        
        # 1. numeric stats
        num_cols = [c for c in df.columns if c.startswith("pre_") or c == "rn"]

        agg_expr = []
        for c in num_cols:
            agg_expr += [
                F.mean(c).alias(f"{c}_mean"),
                F.max(c).alias(f"{c}_max"),
                F.min(c).alias(f"{c}_min"),
                F.stddev(c).alias(f"{c}_std"),
                (F.sum(when(col(c) == 0, 1).otherwise(0)) / F.count(c)).alias(f"{c}_zero_ratio"),
                F.count(c).alias(f"{c}_notnull_count"),
                F.expr(f"last({c}) - first({c})").alias(f"{c}_trend"),
                F.expr(f"percentile_approx({c}, 0.5)").alias(f"{c}_median")
            ]

        # 2. categorical sum via vector aggregation
        cat_cols = [c for c in df.columns if c.endswith("_ohe")]
        assembler = VectorAssembler(inputCols=cat_cols, outputCol="cat_vec")
        df = assembler.transform(df)

        # group by id
        agg_df = df.groupBy("id").agg(
            F.collect_list("cat_vec").alias("vectors_list"),
            *agg_expr,
            F.first("flag").alias("flag"),
            F.first("classWeight").alias("classWeight")
        )

        numeric_cols = [c for c in agg_df.columns if c not in ("id", "vectors_list", "flag", "classWeight", "cat_sum")]
        agg_df = agg_df.fillna(0, subset=numeric_cols)

        # sum collected vectors
        def sum_vecs(vecs):
            if not vecs:
                return Vectors.dense([])
            arr = sum(v.toArray() for v in vecs)
            return Vectors.dense(arr)

        sum_udf = udf(sum_vecs, VectorUDT())

        agg_df = agg_df.withColumn("cat_sum", sum_udf("vectors_list"))

        # assemble final features
        final_assembler = VectorAssembler(
            inputCols=["cat_sum"] + [c for c in agg_df.columns if c not in ("id", "vectors_list", "flag", "classWeight", "cat_sum", "cat_sum", "cat_sum")],
            outputCol="features"
        )

        agg_df = final_assembler.transform(agg_df)

        return agg_df.select("id", "features", "flag", "classWeight").cache()

In [14]:
# Определяем категориальные колонки (все enc_* колонки)
categorical_columns = [col for col in train_df.columns if col.startswith('enc_')]
print(f"Категориальных колонок: {len(categorical_columns)}")
print(categorical_columns[:10])  # Показать первые 10

Категориальных колонок: 29
['enc_paym_0', 'enc_paym_1', 'enc_paym_2', 'enc_paym_3', 'enc_paym_4', 'enc_paym_5', 'enc_paym_6', 'enc_paym_7', 'enc_paym_8', 'enc_paym_9']


In [15]:
# Добавляем таргет
train_target = spark.read.csv('../data/train_target.csv', header=True, inferSchema=True)
train_data_target = train_df.join(train_target, on="id", how="inner")

neg_count = train_data_target.filter(col("flag") == 0).count()
pos_count = train_data_target.filter(col("flag") == 1).count()

class_ratio = neg_count / pos_count  # во сколько раз 0 > 1

train_data_target = train_data_target.withColumn(
    "classWeight",
    when(col("flag") == 1, class_ratio).otherwise(1.0)
)

print(f"Веса добавлены. ratio = {class_ratio:.2f}")

# Шаг 1: Создаем агрегатор и обучаем на train
train_aggregator = PySparkCountAggregator(categorical_columns)
train_features = train_aggregator.fit_transform(train_data_target)

Веса добавлены. ratio = 28.74


# Сохранение DataFrame в формате Parquet

### Сохранение данных из Parquet

In [16]:
database_name = "database"

output_path_train = f"{database_name}/{train_data_name}"
output_path_features = f"{database_name}/train_features"

print(f"Сохраняем данные в: {output_path_train}")

train_df.write.mode("overwrite").option("compression", "snappy").parquet(output_path_train)
train_features.write.mode("overwrite").option("compression", "snappy").parquet(output_path_features)

print("Данные успешно сохранены в Parquet")

Сохраняем данные в: database/train_data
Данные успешно сохранены в Parquet


### Чтение данных из Parquet

In [17]:
df_parquet = spark.read.parquet(output_path_train)

print("Данные успешно загружены из Parquet")
print(f"Количество строк: {df_parquet.count()}")
print(f"Количество колонок: {len(df_parquet.columns)}")

# Показать схему данных
print("Схема данных:")
df_parquet.printSchema()

# Показать первые несколько строк
print("Первые 10 строк:")

df_parquet.limit(10).toPandas().style\
    .set_properties(**{'text-align': 'left', 'max-width': '0', 'white-space': 'nowrap', 'overflow': 'hidden', 'text-overflow': 'ellipsis'})\
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])\
    .format(precision=2)

Данные успешно загружены из Parquet
Количество строк: 26162717
Количество колонок: 61
Схема данных:
root
 |-- id: integer (nullable = true)
 |-- rn: short (nullable = true)
 |-- pre_since_opened: short (nullable = true)
 |-- pre_since_confirmed: short (nullable = true)
 |-- pre_pterm: short (nullable = true)
 |-- pre_fterm: short (nullable = true)
 |-- pre_till_pclose: short (nullable = true)
 |-- pre_till_fclose: short (nullable = true)
 |-- pre_loans_credit_limit: float (nullable = true)
 |-- pre_loans_next_pay_summ: float (nullable = true)
 |-- pre_loans_outstanding: float (nullable = true)
 |-- pre_loans_total_overdue: float (nullable = true)
 |-- pre_loans_max_overdue_sum: float (nullable = true)
 |-- pre_loans_credit_cost_rate: float (nullable = true)
 |-- pre_loans5: byte (nullable = true)
 |-- pre_loans530: byte (nullable = true)
 |-- pre_loans3060: byte (nullable = true)
 |-- pre_loans6090: byte (nullable = true)
 |-- pre_loans90: byte (nullable = true)
 |-- is_zero_loans5: by

,id,rn,pre_since_opened,pre_since_confirmed,pre_pterm,pre_fterm,pre_till_pclose,pre_till_fclose,pre_loans_credit_limit,pre_loans_next_pay_summ,pre_loans_outstanding,pre_loans_total_overdue,pre_loans_max_overdue_sum,pre_loans_credit_cost_rate,pre_loans5,pre_loans530,pre_loans3060,pre_loans6090,pre_loans90,is_zero_loans5,is_zero_loans530,is_zero_loans3060,is_zero_loans6090,is_zero_loans90,pre_util,pre_over2limit,pre_maxover2limit,is_zero_util,is_zero_over2limit,is_zero_maxover2limit,enc_paym_0,enc_paym_1,enc_paym_2,enc_paym_3,enc_paym_4,enc_paym_5,enc_paym_6,enc_paym_7,enc_paym_8,enc_paym_9,enc_paym_10,enc_paym_11,enc_paym_12,enc_paym_13,enc_paym_14,enc_paym_15,enc_paym_16,enc_paym_17,enc_paym_18,enc_paym_19,enc_paym_20,enc_paym_21,enc_paym_22,enc_paym_23,enc_paym_24,enc_loans_account_holder_type,enc_loans_credit_status,enc_loans_credit_type,enc_loans_account_cur,pclose_flag,fclose_flag
0,0,1,18,9,2,3,16,10,11.00,3.00,3.00,0.00,2.00,11.00,6,16,5,4,8,1,1,1,1,1,16.00,2.00,17.00,1,1,1,0,0,3,3,3,3,3,3,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,4,1,0,0
1,0,2,18,9,14,14,12,12,0.00,3.00,3.00,0.00,2.00,11.00,6,16,5,4,8,1,1,1,1,1,16.00,2.00,17.00,1,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,4,1,3,4,1,0,0
2,0,3,18,9,4,8,1,11,11.00,0.00,5.00,0.00,2.00,8.00,6,16,5,4,8,1,1,1,1,1,15.00,2.00,17.00,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,4,1,2,3,1,1,1
3,0,4,4,1,9,12,16,7,12.00,2.00,3.00,0.00,2.00,4.00,6,16,5,4,8,0,1,1,1,1,16.00,2.00,17.00,1,1,1,1,0,0,0,0,0,0,0,0,0,0,1,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,1,1,0,0
4,0,5,5,12,15,2,11,12,10.00,2.00,3.00,0.00,2.00,4.00,6,16,5,4,8,1,1,1,1,1,16.00,2.00,17.00,1,1,1,0,0,0,0,0,0,0,3,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,4,1,0,0
5,0,6,5,0,11,8,12,11,4.00,2.00,3.00,0.00,2.00,4.00,6,16,5,4,8,1,1,1,1,1,9.00,5.00,4.00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,3,4,3,3,3,4,1,2,3,1,0,1
6,0,7,3,9,1,2,12,14,15.00,5.00,3.00,0.00,2.00,3.00,6,16,5,4,8,1,1,1,1,1,16.00,2.00,17.00,1,1,1,0,0,0,0,0,0,0,0,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,4,1,0,0
7,0,8,2,9,2,3,12,14,15.00,5.00,3.00,0.00,2.00,13.00,6,16,5,4,8,1,1,1,1,1,16.00,2.00,17.00,1,1,1,0,0,3,3,3,3,3,3,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,3,4,1,0,0
8,0,9,1,9,11,13,14,8,2.00,5.00,1.00,0.00,2.00,11.00,6,16,5,4,8,1,1,1,1,1,1.00,2.00,17.00,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,3,3,3,3,3,3,4,3,3,3,4,1,2,4,1,0,0
9,0,10,7,9,2,10,8,8,16.00,4.00,2.00,0.00,2.00,11.00,6,16,5,4,8,1,1,1,1,1,15.00,2.00,17.00,0,1,1,0,0,0,0,0,0,3,3,3,3,3,4,3,3,3,3,3,3,3,3,4,3,3,3,4,1,2,4,1,0,0


После успешной записи таблицы останавливаем сессию `Apache Spark`.

In [18]:
spark.stop()